# Loading Data from a probe 
Here we show how to load the data for the specific pretrained probe

In [12]:
from pathlib import Path
import sys, os

# 1) Make your repo importable (so `from misc.probe_data import ProbeData` works)
repo_root = Path("..").resolve()           # <- one level up from notebooks/
sys.path.insert(0, str(repo_root))


In [13]:
from misc.probe_data import ProbeData
probe_name = 'sawmil'
model = 'llama-3-8b'
datapack = 'city_locations'
layer_id = 13
# Per-class readers for OvA:
pb_F = ProbeData(f"{repo_root}/outputs/probes/{probe_name}/{model}/{datapack}_search_task-1/")
pb_T = ProbeData(f"{repo_root}/outputs/probes/{probe_name}/{model}/{datapack}_search_task-0/")
pb_N = ProbeData(f"{repo_root}/outputs/probes/{probe_name}/{model}/{datapack}_search_task-2/")

lp_F = pb_F.load_layer(layer_id)
lp_T = pb_T.load_layer(layer_id)
lp_N = pb_N.load_layer(layer_id)

/Users/carlomarx/Documents/GitHub/trillema-of-truth/outputs/probes/sawmil/llama-3-8b/city_locations_search_task-1
/Users/carlomarx/Documents/GitHub/trillema-of-truth/outputs/probes/sawmil/llama-3-8b/city_locations_search_task-0
/Users/carlomarx/Documents/GitHub/trillema-of-truth/outputs/probes/sawmil/llama-3-8b/city_locations_search_task-2


In [14]:
# probes/ova_projector.py
from __future__ import annotations
import numpy as np

class OvAProjector:
    '''One-vs-All Projector for scoring bags against multiple classes.'''
    def __init__(self, probe_data_dict: dict[int, "ProbeData"], layer_id: int):
        """
        This objects takes a instances of ProbeData 
        probe_data_by_class: {class_id: ProbeData}
        layer_id: which layer’s params to use
        """
        self.order = sorted(probe_data_dict.keys())
        # cache layer params once
        self.params = {c: probe_data_dict[c].load_layer(layer_id) for c in self.order}

    @staticmethod
    def _sanitize(X, val=1e4):
        X = np.asarray(X, dtype=np.float64)
        X = np.nan_to_num(X, nan=0.0, posinf=val, neginf=-val)
        return np.clip(X, -val, val)

    @staticmethod
    def _score_instances(X, direction, bias):
        # X: (n,d), direction: (d,), bias: scalar -> (n,)
        return X @ direction + bias

    def score_bag(self, bag: np.ndarray, pool: str = "max") -> np.ndarray:
        """
        bag: (n,d) -> (C,) scores (one per class)
        pool: 'last' | 'max' | 'mean'
        """
        bag = self._sanitize(bag)
        out = np.zeros(len(self.order), dtype=np.float64)

        for j, c in enumerate(self.order):
            lp = self.params[c]
            # If you saved a full model instead of direction/bias, you could branch here.
            Xs = lp.scaler.transform(bag) if lp.scaler is not None else bag
            s = self._score_instances(Xs, lp.direction, lp.bias)  # (n,)

            if pool == "last":
                v = s[-1]
            elif pool == "max":
                v = float(np.max(s))
            elif pool == "mean":
                v = float(np.mean(s))
            else:
                raise ValueError(f"Unknown pool={pool}")
            out[j] = v

        return out  # (C,)

    def score_bags(self, bags: list[np.ndarray], pool: str = "last") -> np.ndarray:
        return np.stack([self.score_bag(b, pool=pool) for b in bags], axis=0)  # (N,C)


In [15]:
layer_id = 12  # pick a real one
projector = OvAProjector({0: pb_F, 1: pb_T, 2: pb_N}, layer_id=layer_id)


TypeError: float() argument must be a string or a real number, not 'LayerParams'